In [1]:
import pymysql
from configparser import ConfigParser
# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    sql = """
        CREATE DATABASE IF NOT EXISTS testdb;
    """
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute("SHOW DATABASES;")
    result = cursor.fetchall()
    print(f"Databases: {result}")

    sql = """
        CREATE TABLE IF NOT EXISTS testdb.user (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            age INT NOT NULL,
            username VARCHAR(255) NOT NULL UNIQUE,
            password VARCHAR(255) NOT NULL
        );
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES FROM testdb;")
    result = cursor.fetchall()
    print(f"Tables: {result}")


Databases: [{'Database': 'chapter2'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'sys'}, {'Database': 'testdb'}, {'Database': 'transaction_test'}, {'Database': 'world'}]
Tables: [{'Tables_in_testdb': 'user'}]


In [2]:
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database="testdb",
)

with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ("Pochita", 10, "pochita", "123456")
    """
    cursor.execute(sql)

    cursor.execute("SELECT * FROM user")
    result = cursor.fetchall()
    for row in result:
        print(row)


{'id': 1, 'name': 'Pochita', 'age': 10, 'username': 'pochita', 'password': '123456'}


In [3]:
connection.commit() # 提交資料庫變更

In [7]:
def check_user_passowrd(username, password):
    with connection.cursor() as cursor:
        sql = f"""
            SELECT * FROM user WHERE username = '{username}' AND password = '{password}'
        """
        cursor.execute(sql)
        result = cursor.fetchall()

    if result == ():
        print("帳號密碼錯誤")
    else:
        print(result)

check_user_passowrd('pochita', '123456')

[{'id': 1, 'name': 'Pochita', 'age': 10, 'username': 'pochita', 'password': '123456'}]


In [8]:
import random
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES (%s, %s, %s, %s);
    """
    for i in range(10):
        # 隨機生成使用者名稱、年齡和密碼
        user = f'user_{i}'
        username = f'user_{i}'
        age = random.randint(18, 65)
        password = f'password_{random.randint(1000, 10000)}'
        cursor.execute(sql, (user, age, username, password))
    
    connection.commit()

##### ⚠️ <span style="color:red; font-weight:bold">勿使用 f-string 直接拼接 SQL 查詢</span>

In [ ]:
username = 'pochita'
password = '123456'

with connection.cursor() as cursor:
    # 查詢所有使用者
    sql = f"SELECT * FROM user WHERE username='{username}' and password='{password}';"
    cursor.execute(sql)
    result = cursor.fetchall()
    for row in result:
        print(f"\033[32m{row}\033[0m")

In [ ]:
from pprint import pprint
# ❌錯誤的寫法
def check_user_passowrd(username, password):
    with connection.cursor() as cursor:
        sql = f"""
            SELECT * FROM user WHERE username = '{username}' AND password = '{password}'
        """
        # SELECT * FROM user WHERE username = '' OR 1=1; -- ' AND password = '{password}'
        cursor.execute(sql)
        result = cursor.fetchall()

    if result == ():
        print("帳號密碼錯誤")
    else:
        pprint(result)

check_user_passowrd("' OR 1=1; -- ", '')

[{'age': 10,
  'id': 1,
  'name': 'Pochita',
  'password': '123456',
  'username': 'pochita'},
 {'age': 49,
  'id': 2,
  'name': 'user_0',
  'password': 'password_9247',
  'username': 'user_0'},
 {'age': 34,
  'id': 3,
  'name': 'user_1',
  'password': 'password_3661',
  'username': 'user_1'},
 {'age': 31,
  'id': 4,
  'name': 'user_2',
  'password': 'password_2277',
  'username': 'user_2'},
 {'age': 22,
  'id': 5,
  'name': 'user_3',
  'password': 'password_6011',
  'username': 'user_3'},
 {'age': 53,
  'id': 6,
  'name': 'user_4',
  'password': 'password_1992',
  'username': 'user_4'},
 {'age': 62,
  'id': 7,
  'name': 'user_5',
  'password': 'password_7747',
  'username': 'user_5'},
 {'age': 18,
  'id': 8,
  'name': 'user_6',
  'password': 'password_1911',
  'username': 'user_6'},
 {'age': 63,
  'id': 9,
  'name': 'user_7',
  'password': 'password_8496',
  'username': 'user_7'},
 {'age': 19,
  'id': 10,
  'name': 'user_8',
  'password': 'password_9151',
  'username': 'user_8'},
 {'a

In [ ]:
# 正確寫法
def check_user_passowrd(username, password):
    with connection.cursor() as cursor:
        sql = """
            SELECT * FROM user WHERE username = %s AND password = %s
        """
        cursor.execute(sql, (username, password))
        result = cursor.fetchall()

    if result == ():
        print("帳號密碼錯誤")
    else:
        pprint(result)

check_user_passowrd("' OR 1=1; -- ", '')

帳號密碼錯誤


In [ ]:
# 使用 SQL Injection 攻擊，取得所有使用者的資料
username = "' OR 1=1; -- "
password = ""

with connection.cursor() as cursor:
    sql = f"""
        SELECT * FROM user WHERE username = '{username}' AND password = '{password}'
    """
    print(sql)
    cursor.execute(sql)
    result = cursor.fetchall()

    for row in result: 
        print(f"\033[31m{row}\033[0m")

In [ ]:
# 使用 SQL Injection 攻擊，取得所有使用者的資料
username = "' OR 1=1; -- "
password = ""

username = 'pochita'
password = '123456'

with connection.cursor() as cursor:
    sql = """
        SELECT * FROM user WHERE username = %s AND password = %s
    """
    print(sql)
    # 把實際的參數值 放在 execute(<sql語句>, (<參數值>)) 第二個參數
    cursor.execute(sql, (username, password))
    result = cursor.fetchall()
    
print(f"\033[31m{result}\033[0m")


        SELECT * FROM user WHERE username = %s AND password = %s
    


TypeError: not enough arguments for format string

### 🚮清除資源

In [ ]:
with connection.cursor() as cursor:
    sql = """
        DROP DATABASE IF EXISTS testdb;
    """
    cursor.execute(sql)

    cursor.execute("SHOW DATABASES;")
    result = cursor.fetchall()
    print(f"Databases: {result}")